# 06 — LIME/XAI for Hyperspectral Ensemble (15-channel)

**Problem with old notebook:** LIME expects 3-channel images.  
**Solution:** Wrap predict function to handle 15-channel internally.  
**Visualization:** Map PCA bands to pseudo-RGB for display.  

---
Uses models saved by Notebook 05:
- `hsi_custom_cnn.h5`
- `hsi_resnet50.h5`
- `hsi_densenet121.h5`
- `hsi_ensemble_weights.npy`
- `hsi_label_encoder.pkl`

In [ ]:
# ─────────────────────────────────────────────
# CELL 1 — Imports
# ─────────────────────────────────────────────
import os
import numpy as np
import pickle
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cv2

import tensorflow as tf
from tensorflow.keras.models import load_model

from lime import lime_image
from skimage.segmentation import mark_boundaries
from sklearn.metrics import classification_report

print("All imports OK!")
print(f"TensorFlow: {tf.__version__}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 2 — Paths & Config
# ─────────────────────────────────────────────
PROCESSED_DIR = '../data/processed/'
MODEL_DIR     = '../models/'

CNN_PATH      = os.path.join(MODEL_DIR, 'hsi_custom_cnn.h5')
RESNET_PATH   = os.path.join(MODEL_DIR, 'hsi_resnet50.h5')
DENSE_PATH    = os.path.join(MODEL_DIR, 'hsi_densenet121.h5')
WEIGHTS_PATH  = os.path.join(MODEL_DIR, 'hsi_ensemble_weights.npy')
ENCODER_PATH  = os.path.join(MODEL_DIR, 'hsi_label_encoder.pkl')

CLASS_NAMES   = ['Healthy (0.0)', 'Partial (0.5)', 'Deficient (1.0)']

print("Config set.")

In [ ]:
# ─────────────────────────────────────────────
# CELL 3 — Load Data & Models
# ─────────────────────────────────────────────
print("Loading test data (15-channel)...")
X_test = np.load(os.path.join(PROCESSED_DIR, 'X_test_hyper.npy'))   # (318, 128, 128, 15)
y_test = np.load(os.path.join(PROCESSED_DIR, 'y_test_hyper.npy'))

with open(ENCODER_PATH, 'rb') as f:
    le = pickle.load(f)
y_test_enc = le.transform(y_test)

print(f"X_test shape: {X_test.shape}")
assert X_test.shape[-1] == 15, "ERROR: Expected 15-channel data!"

# Load models
print("\nLoading models...")
cnn_model      = load_model(CNN_PATH)
resnet_model   = load_model(RESNET_PATH)
densenet_model = load_model(DENSE_PATH)
ensemble_w     = np.load(WEIGHTS_PATH)

print(f"✅ Models loaded. Ensemble weights: {ensemble_w}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 4 — LIME Predict Function
# 
# KEY FIX: LIME internally converts any image to 3-channel
# We need to:
#   1. Accept 3-channel images from LIME
#   2. Expand them back to 15-channel using PCA reconstruction (approx)
#   3. Feed to all 3 models → ensemble probabilities
# ─────────────────────────────────────────────

# First compute pseudo-RGB display image (for LIME input)
# We use PCA bands 0,1,2 as R,G,B approximation for LIME
def hsi_to_pseudoRGB(hsi_image_15ch):
    """
    Convert 15-channel HSI to 3-channel pseudo-RGB for LIME.
    Uses first 3 PCA components (most variance).
    Output is normalized to [0, 1] float.
    """
    rgb = hsi_image_15ch[:, :, :3].copy()
    # Normalize each channel to [0, 1]
    for c in range(3):
        ch = rgb[:, :, c]
        vmin, vmax = ch.min(), ch.max()
        if vmax > vmin:
            rgb[:, :, c] = (ch - vmin) / (vmax - vmin)
    return rgb.astype(np.float32)


def make_lime_predict_fn(original_15ch_image):
    """
    Creates a LIME-compatible predict function for a 15-channel HSI image.
    
    Strategy:
    - LIME perturbs the pseudo-RGB (3ch) display image
    - For each perturbed 3ch image, we reconstruct an approximate 15ch image:
        * Copy original image
        * Use LIME's mask to zero out masked regions (same as LIME does internally)
        * Feed 15ch image to ensemble
    """
    def predict_fn(perturbed_images_3ch):
        """
        perturbed_images_3ch: numpy array of shape (N, H, W, 3) — from LIME
        returns: numpy array of shape (N, 3) — class probabilities
        """
        N = len(perturbed_images_3ch)
        batch_15ch = np.zeros((N, 128, 128, 15), dtype=np.float32)

        for i, img_3ch in enumerate(perturbed_images_3ch):
            # Detect which pixels LIME masked (set to 0 or grey)
            # LIME hides segments by setting them to hide_color
            # We use the same mask on all 15 channels
            mask_3ch = img_3ch.copy()
            reconstructed = original_15ch_image.copy()

            # Where all 3 channels are near 0 or near grey → masked region
            # Approximate: if pixel is very different from pseudo-RGB, it's masked
            pseudo_rgb = hsi_to_pseudoRGB(original_15ch_image)
            diff = np.abs(mask_3ch - pseudo_rgb).mean(axis=-1)  # (H, W)
            masked_pixels = diff > 0.1  # Threshold for 'masked' pixel

            # Zero out masked pixels across all 15 channels
            reconstructed[masked_pixels] = 0.0
            batch_15ch[i] = reconstructed

        # Ensemble predict
        p_cnn    = cnn_model.predict(batch_15ch, batch_size=32, verbose=0)
        p_resnet = resnet_model.predict(batch_15ch, batch_size=32, verbose=0)
        p_dense  = densenet_model.predict(batch_15ch, batch_size=32, verbose=0)
        ensemble = np.average([p_cnn, p_resnet, p_dense], axis=0, weights=ensemble_w)
        return ensemble

    return predict_fn


print("✅ LIME predict function defined.")
print("   → Uses full 15-channel ensemble internally")
print("   → Displays pseudo-RGB (PCA bands 0,1,2) for visualization")

In [ ]:
# ─────────────────────────────────────────────
# CELL 5 — Run LIME on One Sample
# ─────────────────────────────────────────────
IMAGE_INDEX = 5  # Change this to explore different samples

# Get the 15-channel image
sample_15ch = X_test[IMAGE_INDEX]                         # (128, 128, 15)
true_label  = CLASS_NAMES[y_test_enc[IMAGE_INDEX]]

# Convert to pseudo-RGB for LIME
sample_pseudo_rgb = hsi_to_pseudoRGB(sample_15ch)          # (128, 128, 3) float [0,1]

# Get ensemble prediction
p = np.average([
    cnn_model.predict(sample_15ch[np.newaxis], verbose=0),
    resnet_model.predict(sample_15ch[np.newaxis], verbose=0),
    densenet_model.predict(sample_15ch[np.newaxis], verbose=0)
], axis=0, weights=ensemble_w)[0]

pred_idx        = np.argmax(p)
pred_label      = CLASS_NAMES[pred_idx]
pred_confidence = p[pred_idx] * 100

print(f"Sample index   : {IMAGE_INDEX}")
print(f"True label     : {true_label}")
print(f"Predicted      : {pred_label} ({pred_confidence:.1f}% confidence)")
print(f"All probs      : {dict(zip(CLASS_NAMES, [f'{v:.3f}' for v in p]))}")
print()

# Create predict function for this image
lime_predict_fn = make_lime_predict_fn(sample_15ch)

# Run LIME
print("🔍 Running LIME explainer (this takes ~1-2 minutes)...")
explainer = lime_image.LimeImageExplainer(random_state=42)
explanation = explainer.explain_instance(
    sample_pseudo_rgb,          # 3-channel pseudo-RGB image
    lime_predict_fn,            # Our wrapper that uses 15ch internally
    top_labels=3,
    hide_color=0,
    num_samples=1000,           # Increase to 2000 for research paper quality
    batch_size=32
)
print("✅ LIME explanation complete!")

In [ ]:
# ─────────────────────────────────────────────
# CELL 6 — Visualize LIME Explanation
# ─────────────────────────────────────────────
fig = plt.figure(figsize=(18, 6))
gs  = gridspec.GridSpec(1, 4, figure=fig)

# ── Panel 1: Original pseudo-RGB ──
ax1 = fig.add_subplot(gs[0])
ax1.imshow(sample_pseudo_rgb)
ax1.set_title(f'Pseudo-RGB Image\n(True: {true_label})', fontweight='bold')
ax1.axis('off')

# ── Panel 2: LIME positive regions (predicted class) ──
temp, mask = explanation.get_image_and_mask(
    pred_idx, positive_only=True, num_features=5, hide_rest=False
)
ax2 = fig.add_subplot(gs[1])
ax2.imshow(mark_boundaries(temp, mask, color=(0, 1, 0)))
ax2.set_title(f'LIME — Supporting Regions\n(Pred: {pred_label})', fontweight='bold')
ax2.axis('off')

# ── Panel 3: LIME positive + negative ──
temp2, mask2 = explanation.get_image_and_mask(
    pred_idx, positive_only=False, num_features=10, hide_rest=False
)
ax3 = fig.add_subplot(gs[2])
ax3.imshow(mark_boundaries(temp2, mask2, color=(1, 0, 0)))
ax3.set_title('LIME — For (+) vs Against (-)', fontweight='bold')
ax3.axis('off')

# ── Panel 4: Feature importance bar chart ──
ax4 = fig.add_subplot(gs[3])
vals = explanation.local_exp[pred_idx]
vals_sorted = sorted(vals, key=lambda x: abs(x[1]), reverse=True)[:8]
features, importances = zip(*vals_sorted)
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in importances]
ax4.barh(range(len(features)), importances, color=colors)
ax4.set_yticks(range(len(features)))
ax4.set_yticklabels([f'Region {f}' for f in features])
ax4.set_xlabel('Importance')
ax4.set_title('Top Spectral Regions', fontweight='bold')
ax4.axvline(x=0, color='black', linestyle='-', linewidth=0.5)

plt.suptitle(
    f'LIME Explanation | Predicted: {pred_label} ({pred_confidence:.1f}%) | True: {true_label}',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, f'lime_explanation_sample_{IMAGE_INDEX}.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ LIME visualization saved.")

In [ ]:
# ─────────────────────────────────────────────
# CELL 7 — Run LIME on Multiple Samples (One per class)
# For Research Paper — shows XAI works for all 3 classes
# ─────────────────────────────────────────────
print("Finding one sample from each class...")

# Find indices of samples from each class
class_samples = {}
for cls_idx in range(3):
    candidates = np.where(y_test_enc == cls_idx)[0]
    class_samples[cls_idx] = candidates[0]  # First sample from each class

print(f"Sample indices: {class_samples}")

fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for row, (cls_idx, img_idx) in enumerate(class_samples.items()):
    sample_15ch_i = X_test[img_idx]
    pseudo_rgb_i  = hsi_to_pseudoRGB(sample_15ch_i)
    true_lbl_i    = CLASS_NAMES[cls_idx]

    # Ensemble predict
    p_i = np.average([
        cnn_model.predict(sample_15ch_i[np.newaxis], verbose=0),
        resnet_model.predict(sample_15ch_i[np.newaxis], verbose=0),
        densenet_model.predict(sample_15ch_i[np.newaxis], verbose=0)
    ], axis=0, weights=ensemble_w)[0]
    pred_idx_i  = np.argmax(p_i)
    pred_lbl_i  = CLASS_NAMES[pred_idx_i]

    # Run LIME
    print(f"\nRunning LIME for {true_lbl_i} sample (index {img_idx})...")
    lime_fn_i = make_lime_predict_fn(sample_15ch_i)
    exp_i = explainer.explain_instance(
        pseudo_rgb_i, lime_fn_i,
        top_labels=3, hide_color=0, num_samples=800, batch_size=32
    )

    # Col 0: Original
    axes[row, 0].imshow(pseudo_rgb_i)
    axes[row, 0].set_title(f'True: {true_lbl_i}', fontweight='bold')
    axes[row, 0].axis('off')

    # Col 1: LIME positive
    t, m = exp_i.get_image_and_mask(pred_idx_i, positive_only=True, num_features=5, hide_rest=False)
    axes[row, 1].imshow(mark_boundaries(t, m, color=(0,1,0)))
    axes[row, 1].set_title(f'Pred: {pred_lbl_i}\nGreen = Supporting', fontweight='bold')
    axes[row, 1].axis('off')

    # Col 2: LIME + / -
    t2, m2 = exp_i.get_image_and_mask(pred_idx_i, positive_only=False, num_features=10, hide_rest=False)
    axes[row, 2].imshow(mark_boundaries(t2, m2, color=(1,0,0)))
    axes[row, 2].set_title('Green=For / Red=Against', fontweight='bold')
    axes[row, 2].axis('off')

plt.suptitle('LIME Explanations — One Sample per Class\n(Hyperspectral Ensemble)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'lime_all_classes.png'), dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Multi-class LIME visualization saved as lime_all_classes.png")

In [ ]:
# ─────────────────────────────────────────────
# CELL 8 — PCA Band Importance Analysis
# Which of the 15 PCA components matter most?
# ─────────────────────────────────────────────
print("Analyzing importance of each PCA band via occlusion...")

# Use 20 random test samples
np.random.seed(42)
sample_indices = np.random.choice(len(X_test), size=20, replace=False)

band_importance = np.zeros(15)

for idx in sample_indices:
    original = X_test[idx][np.newaxis]  # (1, 128, 128, 15)
    
    # Baseline prediction
    base_prob = np.average([
        cnn_model.predict(original, verbose=0),
        resnet_model.predict(original, verbose=0),
        densenet_model.predict(original, verbose=0)
    ], axis=0, weights=ensemble_w)[0]
    true_cls = y_test_enc[idx]

    # Occlude each band
    for band in range(15):
        occluded = original.copy()
        occluded[:, :, :, band] = 0.0  # Zero out this band
        occ_prob = np.average([
            cnn_model.predict(occluded, verbose=0),
            resnet_model.predict(occluded, verbose=0),
            densenet_model.predict(occluded, verbose=0)
        ], axis=0, weights=ensemble_w)[0]
        # Importance = drop in correct class probability
        band_importance[band] += (base_prob[true_cls] - occ_prob[true_cls])

band_importance /= len(sample_indices)  # Average

# Plot
plt.figure(figsize=(12, 5))
bars = plt.bar(range(15), band_importance,
               color=['#e74c3c' if v > 0 else '#3498db' for v in band_importance])
plt.xlabel('PCA Component (Band Index)', fontsize=12)
plt.ylabel('Mean Importance (Prob Drop)', fontsize=12)
plt.title('PCA Band Importance via Occlusion Analysis\n(Higher = More Important for correct prediction)',
          fontsize=13, fontweight='bold')
plt.xticks(range(15), [f'PC{i+1}' for i in range(15)])
plt.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'pca_band_importance.png'), dpi=150)
plt.show()

top3 = np.argsort(band_importance)[::-1][:3]
print(f"\nTop 3 most important PCA components: PC{top3[0]+1}, PC{top3[1]+1}, PC{top3[2]+1}")
print("✅ Band importance plot saved as pca_band_importance.png")

In [ ]:
# ─────────────────────────────────────────────
# CELL 9 — Streamlit Helper Function
# Save this function so app.py can call LIME directly
# ─────────────────────────────────────────────
streamlit_lime_code = '''
# ── lime_utils.py ── (place in src/ folder)
import numpy as np
import matplotlib.pyplot as plt
from lime import lime_image
from skimage.segmentation import mark_boundaries

def hsi_to_pseudoRGB(hsi_image_15ch):
    """Convert 15-channel HSI to 3-channel pseudo-RGB."""
    rgb = hsi_image_15ch[:, :, :3].copy().astype(np.float32)
    for c in range(3):
        ch = rgb[:, :, c]
        vmin, vmax = ch.min(), ch.max()
        if vmax > vmin:
            rgb[:, :, c] = (ch - vmin) / (vmax - vmin)
    return rgb

def get_lime_explanation(hsi_image_15ch, models_list, weights,
                          class_names, pred_idx, num_samples=500):
    """
    hsi_image_15ch : numpy array (128, 128, 15)
    models_list    : [cnn_model, resnet_model, dense_model]
    weights        : ensemble weights array
    Returns        : matplotlib figure
    """
    pseudo_rgb = hsi_to_pseudoRGB(hsi_image_15ch)

    def predict_fn(imgs_3ch):
        N = len(imgs_3ch)
        batch = np.zeros((N, 128, 128, 15), dtype=np.float32)
        original_prgb = hsi_to_pseudoRGB(hsi_image_15ch)
        for i, img in enumerate(imgs_3ch):
            recon = hsi_image_15ch.copy()
            diff  = np.abs(img - original_prgb).mean(axis=-1)
            recon[diff > 0.1] = 0.0
            batch[i] = recon
        preds = [m.predict(batch, verbose=0) for m in models_list]
        return np.average(preds, axis=0, weights=weights)

    explainer   = lime_image.LimeImageExplainer(random_state=42)
    explanation = explainer.explain_instance(
        pseudo_rgb, predict_fn,
        top_labels=len(class_names), hide_color=0,
        num_samples=num_samples, batch_size=32
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    t, m = explanation.get_image_and_mask(pred_idx, positive_only=True, num_features=5, hide_rest=False)
    axes[0].imshow(mark_boundaries(t, m, color=(0,1,0)))
    axes[0].set_title(f"Supporting regions\n(Green = important)", fontweight="bold")
    axes[0].axis("off")

    t2, m2 = explanation.get_image_and_mask(pred_idx, positive_only=False, num_features=10, hide_rest=False)
    axes[1].imshow(mark_boundaries(t2, m2, color=(1,0,0)))
    axes[1].set_title("Green=For / Red=Against", fontweight="bold")
    axes[1].axis("off")

    plt.suptitle(f"LIME — Predicted: {class_names[pred_idx]}",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    return fig
'''

# Save lime_utils.py
utils_path = '../src/lime_utils.py'
os.makedirs('../src', exist_ok=True)
with open(utils_path, 'w') as f:
    f.write(streamlit_lime_code.strip())

print(f"✅ Streamlit helper saved to: {utils_path}")
print("\nIn your Streamlit app, use it like:")
print("  from src.lime_utils import get_lime_explanation, hsi_to_pseudoRGB")
print("  fig = get_lime_explanation(processed_img_15ch, [cnn, resnet, dense], weights, CLASS_NAMES, pred_idx)")
print("  st.pyplot(fig)")